# NB4 — Embeddings denses classiques et embeddings de phrases

Notebook des pipelines **P16 à P20**.

## Portée du notebook

Ce notebook fait la transition entre les représentations sparse et les représentations denses préentraînées :
- embeddings statiques orientés tweets ;
- embeddings sous-mots ;
- embeddings de phrases contextuels.

Comme ces modèles peuvent être plus lents à initialiser, il est préférable de les lancer **après** les baselines sparse.

In [ ]:
# Pour un run sur Colab

''' 
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)

'''

In [2]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

from collections import OrderedDict

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    evaluate_sklearn_pipeline,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
)

seed_everything(42)
import mlflow


from mlflow_utils import (
    setup_mlflow_tracking,
    fit_evaluate_and_log_sklearn_pipeline,
)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB4_classical_sentence_embeddings"
RESULTS_DIR = "../../outputs/NB4"

# Configuration MLflow
from pathlib import Path
MLFLOW_EXPERIMENT_NAME = "DT_NB4_classical_sentence_embeddings"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/06 22:28:22 INFO mlflow.tracking.fluent: Experiment with name 'DT_NB4_classical_sentence_embeddings' does not exist. Creating a new experiment.


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB4_classical_sentence_embeddings


In [4]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [5]:
pipelines = OrderedDict({
    "P16_GloVeTwitterMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P17_GloVeTwitterMean_LinearSVC": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P18_FastTextMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="fasttext-wiki-news-subwords-300", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P19_SentenceTransformer_LogReg": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P20_SentenceTransformer_LinearSVC": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LinearSVC(C=1.0)),
    ]),
})

In [ ]:
resultats = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    metrics = fit_evaluate_and_log_sklearn_pipeline(
        name=nom_pipeline,
        estimator=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        notebook_name="NB4",
        family_name="classical_sentence_embeddings",
        output_dir=RESULTS_DIR,
        log_model=MLFLOW_LOG_MODEL,
    )
    resultats.append(metrics)

results_df = round_results(pd.DataFrame(resultats))
display(results_df)


Entraînement -> P16_GloVeTwitterMean_LogReg


Pipeline(steps=[('embed', GensimMeanEmbeddingVectorizer(normalize=True)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------


In [ ]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans {RESULTS_DIR}")